# 02 — Quality control and 4-minute temporal regularization

This notebook aggregates duplicate timestamps, maps the validated observations to a uniform 4-minute grid, applies physical-range checks, and interpolates only internal gaps of at most 12 minutes. BME280 relative humidity is not interpolated because saturation was identified during sensor evaluation.

**Input**

- `data/processed/greenhouse_sensor_data_validated.csv`

**Main outputs**

- `data/processed/greenhouse_sensor_data_4min.csv`
- Quality-control tables in `results/quality_control/`
- Diagnostic figures in PNG and PDF format in `figures/quality_control/`

All paths are relative to the repository root.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "processed").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Run this notebook from the repository root "
        "or from its notebooks directory."
    )

PROJECT_ROOT = find_project_root()
INPUT_FILE = PROJECT_ROOT / "data" / "processed" / "greenhouse_sensor_data_validated.csv"
OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "greenhouse_sensor_data_4min.csv"
RESULTS_DIR = PROJECT_ROOT / "results" / "quality_control"
FIGURES_DIR = PROJECT_ROOT / "figures" / "quality_control"

for directory in (RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

FREQUENCY = "4min"
MAX_INTERPOLATION_BINS = 3

print(f"Input: {INPUT_FILE.relative_to(PROJECT_ROOT)}")


## 1. Load and audit the validated observations

The input is produced by `01_sensor_evaluation.ipynb`. Duplicate timestamp rows are retained for the audit and then aggregated by their mean before regularization.


In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Required input file not found: {INPUT_FILE}")

df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
required_columns = [
    "timestamp", "temp_sht31", "rh_sht31", "temp_bme280",
    "rh_bme280_raw", "rh_bme280_clean", "pressure_bme280",
    "rh_bme280_saturated",
]
missing_columns = sorted(set(required_columns).difference(df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
numeric_columns = [
    "temp_sht31", "rh_sht31", "temp_bme280", "rh_bme280_raw",
    "rh_bme280_clean", "pressure_bme280",
]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")
df["rh_bme280_saturated"] = pd.to_numeric(df["rh_bme280_saturated"], errors="coerce").fillna(0).astype(int)

invalid_timestamp_rows = df.loc[df["timestamp"].isna()].copy()
df = df.loc[df["timestamp"].notna()].sort_values("timestamp", kind="stable").reset_index(drop=True)
df["duplicate_timestamp_flag"] = df["timestamp"].duplicated(keep=False)
duplicate_rows = df.loc[df["duplicate_timestamp_flag"]].copy()

timestamp_audit = pd.DataFrame(
    {
        "indicator": [
            "Original records", "Records with valid timestamps",
            "Invalid timestamps", "Duplicate timestamp rows", "Unique timestamps",
        ],
        "value": [
            len(df) + len(invalid_timestamp_rows), len(df), len(invalid_timestamp_rows),
            len(duplicate_rows), df["timestamp"].nunique(),
        ],
    }
)
timestamp_audit.to_csv(RESULTS_DIR / "01_timestamp_audit.csv", index=False)
invalid_timestamp_rows.to_csv(RESULTS_DIR / "02_invalid_timestamps.csv", index=False)
duplicate_rows.to_csv(RESULTS_DIR / "03_duplicate_timestamp_rows.csv", index=False)
display(timestamp_audit)


## 2. Aggregate duplicate timestamps and create the 4-minute grid

Continuous variables are averaged when several records share a timestamp or a 4-minute bin. The grid is anchored to the start of each day.


In [ ]:
aggregation = {column: "mean" for column in numeric_columns}
aggregation.update({"rh_bme280_saturated": "max", "duplicate_timestamp_flag": "max"})
deduplicated = df.groupby("timestamp", as_index=True).agg(aggregation).sort_index()

records_in_bin = deduplicated.resample(FREQUENCY, origin="start_day").size().rename("records_in_bin")
resampled = deduplicated[numeric_columns].resample(FREQUENCY, origin="start_day").mean()
duplicate_in_bin = (
    deduplicated["duplicate_timestamp_flag"]
    .resample(FREQUENCY, origin="start_day")
    .max()
    .reindex(resampled.index)
    .fillna(False)
    .astype(bool)
)
saturation_in_bin = (
    deduplicated["rh_bme280_saturated"]
    .resample(FREQUENCY, origin="start_day")
    .max()
    .reindex(resampled.index)
    .fillna(0)
    .astype(int)
)

regular = pd.DataFrame(index=resampled.index)
regular.index.name = "timestamp"
regular["temp_sht31_raw"] = resampled["temp_sht31"]
regular["rh_sht31_raw"] = resampled["rh_sht31"]
regular["temp_bme280_raw"] = resampled["temp_bme280"]
regular["rh_bme280_raw"] = resampled["rh_bme280_raw"]
regular["pressure_bme280_raw"] = resampled["pressure_bme280"]
regular["records_in_bin"] = records_in_bin.reindex(regular.index, fill_value=0).astype(int)
regular["empty_bin_flag"] = regular["records_in_bin"].eq(0)
regular["multi_record_bin_flag"] = regular["records_in_bin"].gt(1)
regular["duplicate_timestamp_aggregated_flag"] = duplicate_in_bin.astype(int)
regular["rh_bme280_saturated_flag"] = saturation_in_bin

regularization_summary = pd.DataFrame(
    {
        "indicator": [
            "Nominal frequency", "Number of bins", "Bins with at least one record",
            "Empty bins", "Empty bins [%]", "Bins with multiple records",
            "Mean records per bin", "Maximum records per bin",
        ],
        "value": [
            FREQUENCY, len(regular), int(regular["records_in_bin"].gt(0).sum()),
            int(regular["empty_bin_flag"].sum()), float(regular["empty_bin_flag"].mean() * 100),
            int(regular["multi_record_bin_flag"].sum()), regular["records_in_bin"].mean(),
            int(regular["records_in_bin"].max()),
        ],
    }
)
regularization_summary.to_csv(RESULTS_DIR / "04_regularization_summary.csv", index=False)
display(regularization_summary)


## 3. Physical checks and conservative short-gap interpolation

Only missing runs bounded by observed values and no longer than three 4-minute bins are interpolated. Longer gaps remain missing and are excluded later when modeling windows are constructed.


In [ ]:
physical_ranges = {
    "temp_sht31_raw": (-40, 125),
    "rh_sht31_raw": (0, 100),
    "temp_bme280_raw": (-40, 85),
    "rh_bme280_raw": (0, 100),
    "pressure_bme280_raw": (300, 1100),
}
range_rows = []
for variable, (lower, upper) in physical_ranges.items():
    outside = regular[variable].notna() & ~regular[variable].between(lower, upper)
    range_rows.append(
        {
            "variable": variable, "lower_limit": lower, "upper_limit": upper,
            "n_outside_range": int(outside.sum()), "outside_range_pct": float(outside.mean() * 100),
        }
    )
    regular.loc[outside, variable] = np.nan
physical_range_audit = pd.DataFrame(range_rows)
physical_range_audit.to_csv(RESULTS_DIR / "05_physical_range_audit.csv", index=False)

def interpolate_short_internal_gaps(series, max_bins):
    missing = series.isna()
    run_id = missing.ne(missing.shift(fill_value=False)).cumsum()
    run_length = missing.groupby(run_id).transform("sum")
    bounded = series.ffill().notna() & series.bfill().notna()
    fill_mask = missing & run_length.le(max_bins) & bounded
    candidate = series.interpolate(method="time", limit_area="inside")
    cleaned = series.copy()
    cleaned.loc[fill_mask] = candidate.loc[fill_mask]
    return cleaned, fill_mask

interpolation_map = {
    "temp_sht31": "temp_sht31_raw",
    "rh_sht31": "rh_sht31_raw",
    "temp_bme280": "temp_bme280_raw",
    "pressure_bme280": "pressure_bme280_raw",
}
for stem, raw_column in interpolation_map.items():
    regular[f"{stem}_observed_flag"] = regular[raw_column].notna().astype(int)
    clean, interpolated = interpolate_short_internal_gaps(regular[raw_column], MAX_INTERPOLATION_BINS)
    regular[f"{stem}_clean"] = clean
    regular[f"{stem}_interpolated_flag"] = interpolated.astype(int)
    regular[f"{stem}_available_after_qc_flag"] = clean.notna().astype(int)

regular["rh_bme280_observed_flag"] = resampled["rh_bme280_clean"].notna().astype(int)
regular["rh_bme280_clean"] = resampled["rh_bme280_clean"]
regular["rh_bme280_interpolated_flag"] = 0
regular["rh_bme280_available_after_qc_flag"] = regular["rh_bme280_clean"].notna().astype(int)

regular["target_pair_observed_flag"] = (
    regular["temp_sht31_observed_flag"].eq(1) & regular["rh_sht31_observed_flag"].eq(1)
)
regular["target_pair_available_flag"] = (
    regular["temp_sht31_available_after_qc_flag"].eq(1)
    & regular["rh_sht31_available_after_qc_flag"].eq(1)
)
regular["target_pair_interpolated_flag"] = (
    regular["target_pair_available_flag"] & ~regular["target_pair_observed_flag"]
)
regular["quality_status"] = np.select(
    [regular["target_pair_observed_flag"], regular["target_pair_interpolated_flag"]],
    ["observed_complete", "short_gap_interpolated"],
    default="target_missing",
)

interpolation_rows = []
for stem in ["temp_sht31", "rh_sht31", "temp_bme280", "rh_bme280", "pressure_bme280"]:
    interpolation_rows.append(
        {
            "variable": stem, "total_bins": len(regular),
            "observed_bins": int(regular[f"{stem}_observed_flag"].sum()),
            "interpolated_bins": int(regular[f"{stem}_interpolated_flag"].sum()),
            "available_after_qc_bins": int(regular[f"{stem}_available_after_qc_flag"].sum()),
            "missing_after_qc_bins": int(regular[f"{stem}_available_after_qc_flag"].eq(0).sum()),
            "observed_coverage_pct": float(regular[f"{stem}_observed_flag"].mean() * 100),
            "final_coverage_pct": float(regular[f"{stem}_available_after_qc_flag"].mean() * 100),
        }
    )
interpolation_summary = pd.DataFrame(interpolation_rows)
interpolation_summary.to_csv(RESULTS_DIR / "06_interpolation_summary.csv", index=False)
quality_summary = (regular["quality_status"].value_counts().rename_axis("quality_status").reset_index(name="n_bins"))
quality_summary["percentage"] = quality_summary["n_bins"] / len(regular) * 100
quality_summary.to_csv(RESULTS_DIR / "07_target_quality_status.csv", index=False)
display(physical_range_audit)
display(interpolation_summary)
display(quality_summary)


## 4. Abrupt-change review and final file

The review flags identify large changes between consecutive 4-minute bins. They are diagnostic only; flagged values are not removed automatically.


In [ ]:
step_thresholds = {
    "temp_sht31_clean": 3.0,
    "rh_sht31_clean": 10.0,
    "temp_bme280_clean": 3.0,
    "pressure_bme280_clean": 5.0,
}
step_rows = []
for variable, threshold in step_thresholds.items():
    absolute_change = regular[variable].diff().abs()
    flag_column = f"{variable}_step_review_flag"
    regular[flag_column] = absolute_change.gt(threshold).fillna(False)
    step_rows.append(
        {
            "variable": variable,
            "threshold": threshold,
            "n_events": int(regular[flag_column].sum()),
            "percentage": float(regular[flag_column].mean() * 100),
            "maximum_change": float(absolute_change.max()),
            "percentile_99": float(absolute_change.quantile(0.99)),
        }
    )
step_summary = pd.DataFrame(step_rows)
step_summary.to_csv(RESULTS_DIR / "08_abrupt_change_summary.csv", index=False)

output_order = [
    "temp_sht31_raw", "rh_sht31_raw", "temp_bme280_raw", "rh_bme280_raw", "pressure_bme280_raw",
    "temp_sht31_clean", "rh_sht31_clean", "temp_bme280_clean", "rh_bme280_clean", "pressure_bme280_clean",
    "records_in_bin", "empty_bin_flag", "multi_record_bin_flag",
    "duplicate_timestamp_aggregated_flag", "rh_bme280_saturated_flag",
    "temp_sht31_observed_flag", "rh_sht31_observed_flag", "temp_bme280_observed_flag",
    "rh_bme280_observed_flag", "pressure_bme280_observed_flag",
    "temp_sht31_interpolated_flag", "rh_sht31_interpolated_flag", "temp_bme280_interpolated_flag",
    "rh_bme280_interpolated_flag", "pressure_bme280_interpolated_flag",
    "temp_sht31_available_after_qc_flag", "rh_sht31_available_after_qc_flag",
    "temp_bme280_available_after_qc_flag", "rh_bme280_available_after_qc_flag",
    "pressure_bme280_available_after_qc_flag",
    "target_pair_observed_flag", "target_pair_interpolated_flag", "target_pair_available_flag",
    "quality_status",
    "temp_sht31_clean_step_review_flag", "rh_sht31_clean_step_review_flag",
    "temp_bme280_clean_step_review_flag", "pressure_bme280_clean_step_review_flag",
]
final_data = regular[output_order].reset_index()
final_data.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

intervals = final_data["timestamp"].diff().dt.total_seconds().dropna()
final_grid_summary = pd.DataFrame(
    {
        "indicator": [
            "Final rows",
            "Minimum interval [s]",
            "Maximum interval [s]",
            "Median interval [s]",
            "All intervals equal 240 s",
            "Duplicate timestamps",
        ],
        "value": [
            len(final_data),
            intervals.min(),
            intervals.max(),
            intervals.median(),
            bool(intervals.eq(240).all()),
            int(final_data["timestamp"].duplicated().sum()),
        ],
    }
)
final_grid_summary.to_csv(RESULTS_DIR / "09_final_grid_summary.csv", index=False)

display(step_summary)
display(final_grid_summary)


## 5. Coverage diagnostics


In [ ]:
monthly = regular[["target_pair_observed_flag", "target_pair_available_flag"]].resample("MS").mean() * 100
monthly.columns = ["Observed target-pair coverage", "Coverage after quality control"]
monthly.to_csv(RESULTS_DIR / "10_monthly_target_coverage.csv")

fig, axis = plt.subplots(figsize=(10, 5))
monthly.plot(marker="o", ax=axis)
axis.set_xlabel("Month")
axis.set_ylabel("Coverage (%)")
axis.set_ylim(0, 105)
axis.set_title("Monthly target-pair coverage before and after quality control")
axis.legend(loc="best")
fig.tight_layout()
for extension in ("png", "pdf"):
    fig.savefig(FIGURES_DIR / f"01_monthly_target_coverage.{extension}", dpi=300, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(regular.index, regular["temp_sht31_clean"], linewidth=0.7, color="tab:red")
axes[0].set_ylabel("Air temperature (°C)")
axes[0].set_title("Quality-controlled SHT31 target series")
axes[1].plot(regular.index, regular["rh_sht31_clean"], linewidth=0.7, color="tab:blue")
axes[1].set_ylabel("Relative humidity (%RH)")
axes[1].set_xlabel("Date")
fig.tight_layout()
for extension in ("png", "pdf"):
    fig.savefig(FIGURES_DIR / f"02_quality_controlled_targets.{extension}", dpi=300, bbox_inches="tight")
plt.show()

print(f"Regularized dataset: {OUTPUT_FILE.relative_to(PROJECT_ROOT)}")
print(f"Result tables: {RESULTS_DIR.relative_to(PROJECT_ROOT)}")
print(f"Figures: {FIGURES_DIR.relative_to(PROJECT_ROOT)}")
